### Context
split_dataset.py and stastic.py (original files) both operate on 10-second .npz files (from the author's Baidu zip)

Each record contains: 
1. .hea (2 leads, 250 Hz, about 9.2M samples)
2. .dat signal
3. .atr rhythm annotations

There are 23 10 hour double-lead ECGs of interest. 

numpy.savez is used to save a .npz file which is a zipped archive of files named after their variables. 

In [10]:
import wfdb
import numpy as np
import os

In [2]:
record = wfdb.rdrecord('/Users/tristanmartin/Desktop/IIB PROJECT/datasets/MIT-BIH-AFDB/04015')
ann = wfdb.rdann('/Users/tristanmartin/Desktop/IIB PROJECT/datasets/MIT-BIH-AFDB/04015', 'atr')

print(type(record.p_signal))
num_samples, num_leads = np.shape(record.p_signal)
print(f'There are {num_samples} samples with {num_leads} leads each.')
print(f'These are samples at which the {len(ann.aux_note)} annotations occur: {ann.sample}.')
print(f'These are the corresponding labels: {ann.aux_note}.')

<class 'numpy.ndarray'>
There are 9205760 samples with 2 leads each.
These are samples at which the 15 annotations occur: [     30  102584  119604  121773  122194  133348  166857 1096245 1098054
 1135296 1139595 1422436 1423548 1459277 1460416].
These are the corresponding labels: ['(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N', '(AFIB', '(N'].


### Target Format
dataset.py's _load_all_segments() loads {record_id}.npz and expects two arrays: 
1. segments (n_windows, 2500, 1): 10 seconds x 2500 Hz with a single lead
2. labels (n_windows,): contains strings 'N' and 'AFIB'

In [11]:
from scipy.signal import butter, sosfiltfilt
import neurokit2

In [ ]:
# Most of the following was generated by Claude. It is used to convert the raw data files into 
# 10 second segments for use in this repository. The bandpass filtering functions was written by me. 
# The next two cells are used for visualizing segments. 

RAW_DIR = '/Users/tristanmartin/Desktop/IIB PROJECT/datasets/MIT-BIH-AFDB'
OUT_DIR = '/Users/tristanmartin/Desktop/IIB PROJECT/MGCNet-Research/afdb_dataset/data'

FS = 250                # AFDB is already 250 Hz, so no resampling
WIN_LEN = 10 * FS       # 10 s windows -> 2500 samples
LEAD = 0                # ECG1 only; the model is single-lead (in_channels: 1)
KEEP_LABELS = {'N', 'AFIB'}
QUALIFIED = {'Excellent'}   # zhao2018 grades counted as "qualified"; 'Unacceptable' is dropped

# Filtering
def bandpass_signal(record, low_f = 0.5, high_f = 40, order = 4):
    signal = record.p_signal[:, 0]
    fs = record.fs
    sos = butter(order, [low_f, high_f], btype = 'bandpass', fs = fs, output = 'sos')
    return sosfiltfilt(sos, signal)

# Assigns a label to every sample, not just the changes of rhythms. 
# Samples before the first annotation stay ''.

def rhythm_per_sample(ann, sig_len):
    per_sample = np.full(sig_len, '', dtype=object)
    for i, (start, note) in enumerate(zip(ann.sample, ann.aux_note)):
        end = ann.sample[i + 1] if i + 1 < len(ann.sample) else sig_len
        per_sample[start:end] = note.strip('(').strip('\x00')
    return per_sample

# Keep a window only if one rhythm covers all of it and that rhythm is N or AFIB.
# Windows that cross a rhythm change, or contain AFL/J/unlabelled samples, are dropped (returns None).
def label_window(window_labels):
    unique = set(window_labels) # Checks the number of different labels in the window
    if len(unique) != 1:
        return None
    label = unique.pop() # Assigns the sole label in unique to 'label'
    return label if label in KEEP_LABELS else None

# Zhao et al. (2018) signal quality check, as cited in the MGCNet paper, via NeuroKit2.
# Fuses R-peak agreement between two detectors (qSQI), kurtosis (kSQI), QRS-band power (pSQI) and
# baseline wander (basSQI) into 'Excellent' / 'Barely acceptable' / 'Unacceptable'.
# Expects an already bandpassed window. Returns True if the window is "qualified".
def quality_assessment(window):
    try:
        grade = neurokit2.ecg_quality(window, sampling_rate=FS, method='zhao2018', approach='simple')
    except Exception:   # e.g. too few R-peaks found to compute the indices
        return False
    return grade in QUALIFIED

def convert_record(record_id):
    path = os.path.join(RAW_DIR, record_id)
    sig = bandpass_signal(wfdb.rdrecord(path, channels=[LEAD]))
    per_sample = rhythm_per_sample(wfdb.rdann(path, 'atr'), len(sig))

    segments, labels = [], []
    for start in range(0, len(sig) - WIN_LEN + 1, WIN_LEN):   # non-overlapping windows
        label = label_window(per_sample[start:start + WIN_LEN])
        if label is None:
            continue
        window = sig[start:start + WIN_LEN]
        if not quality_assessment(window):
            continue
        segments.append(window[:, None])                         # (2500, 1)
        labels.append(label)

    np.savez(os.path.join(OUT_DIR, record_id + '.npz'),
             segments=np.stack(segments).astype(np.float32),    # (n_windows, 2500, 1)
             labels=np.array(labels))                            # (n_windows,)
    return labels

os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(RAW_DIR, 'RECORDS')) as f:
    records = [r.strip() for r in f if r.strip()]

for record_id in records:
    if not os.path.exists(os.path.join(RAW_DIR, record_id + '.dat')):
        print(f'{record_id}: no .dat file, skipped')   # 00735 and 03665 have annotations only
        continue
    labels = convert_record(record_id)
    print(f'{record_id}: N={labels.count("N")}, AFIB={labels.count("AFIB")}')

00735: no .dat file, skipped
03665: no .dat file, skipped
04015: N=3205, AFIB=16
04043: N=2302, AFIB=212
04048: N=3158, AFIB=6
04126: N=3358, AFIB=116
04746: N=859, AFIB=1920
04908: N=7, AFIB=0
04936: N=633, AFIB=1476
05091: N=3134, AFIB=2
05121: N=1, AFIB=14
05261: N=2881, AFIB=38
06426: N=15, AFIB=50
06453: N=12, AFIB=0
06995: N=1378, AFIB=1296
07162: N=0, AFIB=2261
07859: N=0, AFIB=2207
07879: N=1201, AFIB=2082
07910: N=3026, AFIB=575
08215: N=659, AFIB=2851
08219: N=2630, AFIB=720
08378: N=277, AFIB=9
08405: N=970, AFIB=2561
08434: N=3357, AFIB=139
08455: N=1043, AFIB=2290


In [13]:
# Need to visualize the ECGs because the label_counts.json files are displaying discrepancies
# Visualise N_SAMPLES random 10 s segments from the records in data_split/bihaf.
# Constants are redefined here so the conversion cell above doesn't need re-running.

import json
from collections import Counter
import matplotlib.pyplot as plt
from wfdb import processing

OUT_DIR = '/Users/tristanmartin/Desktop/IIB PROJECT/MGCNet-Research/afdb_dataset/data'
SPLIT_DIR = '/Users/tristanmartin/Desktop/IIB PROJECT/MGCNet-Research/preprocess/data_split/bihaf'
FS = 250
WIN_LEN = 10 * FS
N_SAMPLES = 200
SEED = 0
N_FOLDS = 5

with open(os.path.join(SPLIT_DIR, 'label_counts.json')) as f:
    label_counts = json.load(f)
record_ids = sorted(label_counts.keys())

# Lists every segment as (record_id, segment_index, label). Only the labels are read, so it's cheap.
def build_index(record_ids):
    index = []
    for rid in record_ids:
        with np.load(os.path.join(OUT_DIR, rid + '.npz')) as d:
            labels = d['labels']
        index += [(rid, i, str(lab)) for i, lab in enumerate(labels)]
    return index

# Chooses n entries of the index uniformly at random, without replacement,
# so the sample reflects the dataset's actual mix of records and labels.
def sample_segments(index, n, rng):
    chosen = rng.choice(len(index), size=n, replace=False)
    return [index[k] for k in chosen]

# Loads the chosen segments, opening each record's .npz once. Returns (record_id, segment_index, label, signal).
def load_samples(chosen):
    by_record = {}
    for rid, i, lab in chosen:
        by_record.setdefault(rid, []).append((i, lab))
    samples = []
    for rid, items in sorted(by_record.items()):
        with np.load(os.path.join(OUT_DIR, rid + '.npz')) as d:
            segs = d['segments']
        samples += [(rid, i, lab, segs[i, :, 0]) for i, lab in sorted(items)]
    return samples

index = build_index(record_ids)
samples = load_samples(sample_segments(index, N_SAMPLES, np.random.default_rng(SEED)))

# Number of segments of each type, across all records and in the sample
all_counts = Counter(lab for _, _, lab in index)
sample_counts = Counter(s[2] for s in samples)
print(f'{"Label":<6} {"All":>8} {"Sampled":>8}')
for lab in sorted(all_counts):
    print(f'{lab:<6} {all_counts[lab]:>8,} {sample_counts[lab]:>8,}   ({all_counts[lab] / len(index):.0%} of all)')
print(f'{"Total":<6} {len(index):>8,} {len(samples):>8,}')

# Retained segments per record, counted from the .npz files and checked against label_counts.json
per_record = {rid: Counter() for rid in record_ids}
for rid, _, lab in index:
    per_record[rid][lab] += 1

print(f'\n{"Record":<7} {"N":>6} {"AFIB":>6} {"Total":>6}')
for rid in record_ids:
    n, af = per_record[rid]['N'], per_record[rid]['AFIB']
    check = '' if label_counts[rid] == {'N': n, 'AFIB': af} else '  <- differs from label_counts.json'
    print(f'{rid:<7} {n:>6,} {af:>6,} {n + af:>6,}{check}')

# Records and N / AFIB segments in the train and val split of each fold
print(f'\n{"Fold":<5} {"Train (records / N / AFIB)":>28} {"Val (records / N / AFIB)":>28}')
for k in range(1, N_FOLDS + 1):
    with open(os.path.join(SPLIT_DIR, f'fold_{k}.json')) as f:
        fold = json.load(f)
    cols = []
    for split in ('train', 'val'):
        rids = fold[split]
        n = sum(per_record[r]['N'] for r in rids)
        af = sum(per_record[r]['AFIB'] for r in rids)
        cols.append(f'{len(rids)} / {n:,} / {af:,}')
    print(f'{k:<5} {cols[0]:>28} {cols[1]:>28}')

Label       All  Sampled
AFIB     20,841       72   (38% of all)
N        34,106      128   (62% of all)
Total    54,947      200

Record       N   AFIB  Total
04015    3,205     16  3,221
04043    2,302    212  2,514
04048    3,158      6  3,164
04126    3,358    116  3,474
04746      859  1,920  2,779
04908        7      0      7
04936      633  1,476  2,109
05091    3,134      2  3,136
05121        1     14     15
05261    2,881     38  2,919
06426       15     50     65
06453       12      0     12
06995    1,378  1,296  2,674
07162        0  2,261  2,261
07859        0  2,207  2,207
07879    1,201  2,082  3,283
07910    3,026    575  3,601
08215      659  2,851  3,510
08219    2,630    720  3,350
08378      277      9    286
08405      970  2,561  3,531
08434    3,357    139  3,496
08455    1,043  2,290  3,333

Fold    Train (records / N / AFIB)     Val (records / N / AFIB)
1             18 / 26,616 / 16,593            5 / 7,490 / 4,248
2             18 / 25,877 / 15,555          

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# R-peak locations from wfdb's XQRS detector (as in qrs_locations() in tools/simple_tools.ipynb)
def r_peaks(sig, fs=FS):
    xqrs = processing.XQRS(sig=sig, fs=fs)
    xqrs.detect(verbose=False)
    return xqrs.qrs_inds

# Plots one segment. N is blue, AFIB is red, detected R-peaks are black crosses.
# The title shows the beat count and the RR-interval std (AFIB should give irregular RR intervals).
def plot_one(k, fs=FS):
    rid, i, lab, sig = samples[k]
    t = np.arange(WIN_LEN) / fs
    peaks = r_peaks(sig, fs)
    rr_std = np.std(np.diff(peaks) / fs) * 1000 if len(peaks) > 2 else np.nan
    fig, ax = plt.subplots(figsize=(14, 3.5), constrained_layout=True)
    ax.plot(t, sig, linewidth=0.8, color='tab:red' if lab == 'AFIB' else 'tab:blue')
    ax.plot(peaks / fs, sig[peaks], 'kx', markersize=6)
    ax.set_title(f'[{k + 1}/{len(samples)}]  {rid} #{i} | {lab} | {len(peaks)} beats | RR std {rr_std:.0f} ms')
    ax.set_xlabel('Time (s)')
    plt.show()

# Prev / Next buttons and a slider to click through the samples one at a time.
# continuous_update=False: only redraw when the slider is released, not for every value dragged past.
slider = widgets.IntSlider(value=0, min=0, max=len(samples) - 1, description='Segment', continuous_update=False)
prev_btn = widgets.Button(description='◀ Prev')
next_btn = widgets.Button(description='Next ▶')
out = widgets.Output()

def redraw(change=None):
    with out:
        out.clear_output(wait=True)
        plot_one(slider.value)

prev_btn.on_click(lambda _: setattr(slider, 'value', max(slider.value - 1, slider.min)))
next_btn.on_click(lambda _: setattr(slider, 'value', min(slider.value + 1, slider.max)))
slider.observe(redraw, names='value')

display(widgets.HBox([prev_btn, next_btn, slider]), out)
redraw()